# Myeloid ONLY Merged Pipeline v2.0
**Architecture v2.0 — CellTypist-equal scANVI:**
1. Ref + Query concatenated **symmetrically** (no ref/query label asymmetry)
2. CellTypist annotates **all cells** (ref + query together)
3. scVI batch correction on HVG
4. scANVI trains on **CellTypist labels for all cells** — query is NOT set to Unknown
5. Low-confidence CellTypist cells (< threshold) → `Unknown` for scANVI
6. Reference original labels preserved as `cell_type_fine_ref` (QC only, not used for training)

**Fixes carried from v1.4:**
- `BATCH_KEY` existence + NA-safe prefix before concat
- `symbol_base` matching in CellTypist gene overlap
- `celltypist_proba` explicitly `reindex`-aligned
- `label_order` from `predict(soft=True)` DataFrame columns
- `os.environ` thread vars before `import torch`
- All label columns cast to `category` before `write_h5ad`

## Cell 0 — Imports & Global Settings

In [1]:
import os

import sys, warnings, json, gc, joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
from scipy.stats import entropy

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import scanpy as sc
import scvi
import celltypist
from celltypist import models
from umap import UMAP

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

print("=" * 80)
print("Myeloid ONLY Merged Pipeline v2.0  (CellTypist-equal scANVI)")
print("=" * 80)
gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
scvi.settings.dl_num_workers = 0
import time
import traceback


/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Myeloid ONLY Merged Pipeline v2.0  (CellTypist-equal scANVI)
GPU available: True
GPU: Tesla V100-SXM2-16GB


## Cell 1 — Configuration

In [2]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

REFERENCE_H5AD = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
QUERY_H5AD     = "/home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/myeloid_cells.h5ad"

REF_LABEL_FINE = "cell_type_L3"   # kept as QC reference column only

BATCH_KEY  = "sample"
TISSUE_KEY = "tissue"

OUTPUT_DIR    = "/home/h2048/data/py/20260308/myeloid_only_merged_pipeline_v2"
OUTPUT_PREFIX = "myeloid_merged_v2"

INCLUDE_COARSE_TYPES = None

N_HVG                = 4000
FORCE_MARKERS_IN_HVG = True

SCVI_N_LATENT   = 100
SCVI_N_LAYERS   = 2
SCVI_N_HIDDEN   = 128
SCVI_DROPOUT    = 0.1
MAX_EPOCHS_SCVI = 400

MAX_EPOCHS_SCANVI  = 200
UNLABELED_CATEGORY = "Unknown"

BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 0.0

# CellTypist — annotates ALL merged cells (ref + query equally)
CELLTYPIST_MODEL         = "/home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl"
CELLTYPIST_MAJORITY_VOTE = True

# CellTypist output keys
CELLTYPIST_DIRECT_LABEL_KEY = "celltypist_label_direct"
CELLTYPIST_DIRECT_FILT_KEY  = "celltypist_label_direct_filt"
CELLTYPIST_CONF_THRESHOLD   = 0.5   # cells below this get UNLABELED_CATEGORY for scANVI
CELLTYPIST_SAVE_PROBA       = True

# scANVI label source:
#   "direct"   -> celltypist_label_direct        (unfiltered majority-vote)
#   "filtered" -> celltypist_label_direct_filt   (low-conf -> Unknown)
SCANVI_LABEL_SOURCE = "filtered"   # recommended: use filtered so uncertain cells are Unknown

QUERY_LEIDEN_RESOLUTION = 1.0

# Myeloid signatures
CLASSICAL_MONO_THRESHOLD    = 0.3
NONCLASSICAL_MONO_THRESHOLD = 0.3

MYELOID_CORE_MARKERS      = ["LYZ","CD14","CD33","PTPRC","ITGAM","ITGAX"]
CLASSICAL_MONO_MARKERS    = ["CD14","FCGR1A","CCR2","CD36","SELL"]
NONCLASSICAL_MONO_MARKERS = ["FCGR3A","FCER1G","PICALM","RHOC"]
INTERMEDIATE_MONO_MARKERS = ["CD14","FCGR3A","CD86","HLA-DRA"]
MACROPHAGE_MARKERS        = ["CD68","CD163","MRC1","MARCO","MSR1","FCGR2A","APOE","C1QA"]
M1_MACROPHAGE_MARKERS     = ["CD86","CD80","TNF","IL1B","NOS2","CXCL9","CXCL10"]
M2_MACROPHAGE_MARKERS     = ["CD163","MRC1","ARG1","IL10","TGFB1","CCL22","PPARG"]
CDC1_MARKERS              = ["CLEC9A","XCR1","WDFY4","IRF8","BATF3"]
CDC2_MARKERS              = ["CD1C","FCER1A","CLEC10A","CD2","ESAM"]
PDC_MARKERS               = ["LILRA4","CLEC4C","NRP1","TCF4","IRF7","GZMB"]
NEUTROPHIL_MARKERS        = ["S100A8","S100A9","FCGR3B","CSF3R","CEACAM8","CXCR2","FCN1"]
MAST_CELL_MARKERS         = ["KIT","TPSAB1","TPSB2","HPGDS","MS4A2","FCER1A","CPA3"]
EOSINOPHIL_MARKERS        = ["EPX","PRG2","CLC","CCL26","SIGLEC8"]
PROLIF_MARKERS            = ["MKI67","TOP2A","PCNA"]

STRESS_SIGNATURE_GENES = [
    "HSPA1A","HSPA1B","HSPA8","HSP90AA1","HSP90AB1","DNAJB1",
    "JUN","JUNB","JUND","FOS","FOSB","EGR1","IER2"
]
S_GENES = [
    "MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UHRF1",
    "GINS2","MCM6","CDCA7","DTL","PRIM1","HELLS","RFC2","RPA2",
    "NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7",
    "POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1"
]
G2M_GENES = [
    "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80",
    "CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A",
    "SMC4","CCNB1","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E"
]
FORCED_MARKERS = list(set(
    MYELOID_CORE_MARKERS + CLASSICAL_MONO_MARKERS + NONCLASSICAL_MONO_MARKERS +
    INTERMEDIATE_MONO_MARKERS + MACROPHAGE_MARKERS + M1_MACROPHAGE_MARKERS +
    M2_MACROPHAGE_MARKERS + CDC1_MARKERS + CDC2_MARKERS + PDC_MARKERS +
    NEUTROPHIL_MARKERS + MAST_CELL_MARKERS + EOSINOPHIL_MARKERS + PROLIF_MARKERS
))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sc.settings.seed   = RANDOM_SEED
scvi.settings.seed = RANDOM_SEED

# Visualization & timing constants
PIPELINE_START = time.time()
FIGURE_DPI    = 300
FIGURE_FORMAT = "pdf"
UMAP_SIZE     = 3
UMAP_ALPHA    = 0.6


Seed set to 42


## Cell 2 — Helper Functions

In [3]:
# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def ensure_counts_layer(adata, counts_layer="counts"):
    if counts_layer not in (adata.layers or {}):
        print(f"  [WARN] layers['{counts_layer}'] not found, checking .X...")
        if hasattr(adata, 'X') and adata.X is not None:
            X_s = adata.X[:1000].toarray().flatten() if issparse(adata.X) else np.asarray(adata.X[:1000]).ravel()
            s   = np.asarray(X_s, dtype=np.float64)
            if np.any(s < 0):
                raise ValueError(".X contains negative values!")
            if np.allclose(s, np.round(s), atol=1e-6):
                print(f"  -> Auto-copying .X to layers['{counts_layer}']")
                adata.layers[counts_layer] = csr_matrix(adata.X) if issparse(adata.X) else adata.X.copy()
            else:
                raise ValueError(f"CRITICAL: .X is not integer counts. Range: [{s.min():.4f},{s.max():.4f}]")
        else:
            raise ValueError(f"CRITICAL: layers['{counts_layer}'] not found and .X is None.")

    X_c = adata.layers[counts_layer]
    sd  = X_c.data[:1000] if issparse(X_c) else X_c.flat[:1000]
    s   = np.asarray(sd, dtype=np.float64)
    if s.size == 0:
        print(f"  [WARN] sampled counts empty; skipping integer check")
    else:
        if np.any(s < 0):
            raise ValueError("counts contains negative values!")
        if not np.allclose(s, np.round(s), atol=1e-6):
            raise ValueError(f"counts looks non-integer. Range: [{s.min():.4f},{s.max():.4f}]")
    if issparse(X_c) and not isinstance(X_c, csr_matrix):
        adata.layers[counts_layer] = csr_matrix(X_c)
    return counts_layer


def ensure_batch_tissue(adata, batch_key, tissue_key):
    for key, placeholder in [(batch_key, "unknown_batch"), (tissue_key, "unknown_tissue")]:
        if key not in adata.obs.columns:
            print(f"  [WARN] {key} not found, creating placeholder")
            adata.obs[key] = placeholder
        adata.obs[key] = adata.obs[key].astype("string").fillna(placeholder).astype("category")
        if placeholder not in adata.obs[key].cat.categories:
            adata.obs[key] = adata.obs[key].cat.add_categories([placeholder])


def compute_module_score_efficient(adata, gene_list, score_name, counts_layer="counts"):
    genes    = [g for g in gene_list if g in adata.var_names]
    gene_idx = adata.var_names.get_indexer(genes)
    gene_idx = gene_idx[gene_idx >= 0]
    if len(gene_idx) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return
    adata_tmp = sc.AnnData(X=adata.layers[counts_layer][:, gene_idx].copy(),
                           var=adata.var.iloc[gene_idx].copy())
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)
    X_n = adata_tmp.X
    me  = np.asarray(X_n.mean(axis=1)).flatten() if issparse(X_n) else np.asarray(X_n).mean(axis=1).flatten()
    adata.obs[score_name] = me
    mn, mx = float(me.min()), float(me.max())
    adata.obs[f"{score_name}_norm"] = (me - mn) / (mx - mn) if mx > mn else 0.0
    del adata_tmp; gc.collect()


def compute_monocyte_subtype_scores(adata):
    print("  -> Computing monocyte subtype scores...")
    compute_module_score_efficient(adata, CLASSICAL_MONO_MARKERS,    "Classical_Mono_score")
    compute_module_score_efficient(adata, NONCLASSICAL_MONO_MARKERS, "NonClassical_Mono_score")
    compute_module_score_efficient(adata, INTERMEDIATE_MONO_MARKERS, "Intermediate_Mono_score")
    c_hi = adata.obs["Classical_Mono_score_norm"]    > CLASSICAL_MONO_THRESHOLD
    nc_hi= adata.obs["NonClassical_Mono_score_norm"] > NONCLASSICAL_MONO_THRESHOLD
    im_hi= adata.obs["Intermediate_Mono_score_norm"] > 0.3
    conditions = [c_hi & ~nc_hi, nc_hi & ~c_hi, c_hi & nc_hi & im_hi, c_hi & nc_hi & ~im_hi]
    choices    = ["Classical","NonClassical","Intermediate","DoublePositive"]
    adata.obs["mono_subtype_by_score"] = np.select(conditions, choices, default="Negative")
    for lbl in choices + ["Negative"]:
        print(f"    {lbl}: {(adata.obs['mono_subtype_by_score']==lbl).sum()}")


def prepare_covariates(adata):
    print("  -> Validating counts...")
    ensure_counts_layer(adata, "counts")
    print("  -> Batch/Tissue...")
    ensure_batch_tissue(adata, BATCH_KEY, TISSUE_KEY)
    print("  -> Signature scores...")
    if "pct_counts_mt" not in adata.obs.columns:
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")
    compute_monocyte_subtype_scores(adata)
    compute_module_score_efficient(adata, MACROPHAGE_MARKERS,     "Macrophage_score")
    compute_module_score_efficient(adata, NEUTROPHIL_MARKERS,     "Neutrophil_score")
    compute_module_score_efficient(adata, STRESS_SIGNATURE_GENES, "stress_score")
    if not all(k in adata.obs.columns for k in ["S_score","G2M_score"]):
        s_in = [g for g in S_GENES   if g in adata.var_names]
        g_in = [g for g in G2M_GENES if g in adata.var_names]
        if len(s_in) >= 5 and len(g_in) >= 5:
            cc   = list(dict.fromkeys(s_in + g_in))
            idx  = adata.var_names.get_indexer(cc)
            ad_  = sc.AnnData(X=adata.layers["counts"][:, idx].copy(), var=adata.var.iloc[idx].copy())
            sc.pp.normalize_total(ad_, target_sum=1e4); sc.pp.log1p(ad_)
            sc.tl.score_genes_cell_cycle(ad_, s_genes=s_in, g2m_genes=g_in)
            adata.obs["S_score"]   = ad_.obs["S_score"].values
            adata.obs["G2M_score"] = ad_.obs["G2M_score"].values
            adata.obs["phase"]     = ad_.obs["phase"].values
            del ad_; gc.collect()
        else:
            adata.obs["S_score"] = 0.0; adata.obs["G2M_score"] = 0.0; adata.obs["phase"] = "G1"


def run_celltypist_on_full_genes(adata_merged, model_name=CELLTYPIST_MODEL, majority_vote=True):
    # Annotates ALL merged cells (ref + query) equally.
    # symbol_base matching handles var_names_make_unique suffixes.
    print("\n[CellTypist] Annotating ALL cells (ref + query)...")
    if os.path.isfile(model_name):
        model = models.Model.load(model_name)
        print(f"  -> Loaded from: {model_name}")
    else:
        bn = os.path.basename(model_name)
        models.download_models(model=bn)
        model = models.Model.load(bn)

    model_genes = set(model.features)
    if "symbol_base" in adata_merged.var.columns:
        base_to_var = {}
        for vn, sb in zip(adata_merged.var_names, adata_merged.var["symbol_base"]):
            if sb in model_genes and sb not in base_to_var:
                base_to_var[sb] = vn
        available_genes = list(base_to_var.values())
        print(f"  -> {len(available_genes)}/{len(model_genes)} genes matched via symbol_base")
    else:
        available_genes = [g for g in adata_merged.var_names if g in model_genes]
        print(f"  -> {len(available_genes)}/{len(model_genes)} genes matched via var_names")

    if len(available_genes) < 100:
        raise ValueError(f"Too few overlapping genes ({len(available_genes)}) for CellTypist!")

    adata_ct = adata_merged[:, available_genes].copy()
    if "symbol_base" in adata_ct.var.columns:
        adata_ct.var_names = pd.Index(adata_ct.var["symbol_base"].values)
        adata_ct.var_names_make_unique()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    predictions = celltypist.annotate(adata_ct, model=model, majority_voting=majority_vote, mode="best match")
    pl = predictions.predicted_labels

    if isinstance(pl, pd.DataFrame):
        pred   = pl.get("predicted_labels", pl.iloc[:,0]).astype(str).values
        conf   = predictions.probability_matrix.max(axis=1).values
        maj    = pl["majority_voting"].astype(str).values if majority_vote and "majority_voting" in pl.columns else None
    else:
        pred = pl.astype(str).values
        conf = predictions.probability_matrix.max(axis=1).values
        maj  = None

    adata_merged.obs["celltypist_pred"]       = pred
    adata_merged.obs["celltypist_confidence"] = conf
    if maj is not None:
        adata_merged.obs["celltypist_majority"] = maj

    print("  -> CellTypist complete:")
    print(pd.Series(pred).value_counts().head(10))
    del adata_ct; gc.collect()
    return predictions


def export_celltypist_direct_results(adata_merged, predictions):
    # Writes celltypist_label_direct (unfiltered majority-vote) and
    # celltypist_label_direct_filt (low-conf -> Unknown) for ALL cells.
    # Probability matrix explicitly reindex-aligned to obs_names.
    print("  -> Exporting CellTypist direct branch...")

    src_key = "celltypist_majority" if "celltypist_majority" in adata_merged.obs.columns else "celltypist_pred"
    raw = pd.Series(adata_merged.obs[src_key].astype(str).values, index=adata_merged.obs_names)
    adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY] = raw.astype("category")

    conf = adata_merged.obs["celltypist_confidence"].values
    filt = np.where(conf >= CELLTYPIST_CONF_THRESHOLD, raw.values, UNLABELED_CATEGORY)
    adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY] = pd.Categorical(filt)
    n_low = int((conf < CELLTYPIST_CONF_THRESHOLD).sum())
    print(f"    {CELLTYPIST_DIRECT_LABEL_KEY}: {adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].nunique()} types")
    print(f"    {CELLTYPIST_DIRECT_FILT_KEY}:  {n_low} low-conf cells -> '{UNLABELED_CATEGORY}'")

    if CELLTYPIST_SAVE_PROBA:
        proba_mat = predictions.probability_matrix
        if isinstance(proba_mat, pd.DataFrame):
            proba_df = proba_mat.copy()
        else:
            proba_df = pd.DataFrame(np.asarray(proba_mat, dtype=np.float32), index=adata_merged.obs_names)

        if len(proba_df) != adata_merged.n_obs:
            raise ValueError(f"CellTypist proba row mismatch: {len(proba_df)} vs {adata_merged.n_obs}")
        if not proba_df.index.equals(adata_merged.obs_names):
            proba_df = proba_df.reindex(adata_merged.obs_names)
        if proba_df.isna().any().any():
            raise ValueError("NaN in CellTypist probability matrix after reindex!")

        adata_merged.obsm["celltypist_proba"]      = proba_df.values.astype(np.float32)
        adata_merged.uns["celltypist_label_order"] = [str(x) for x in proba_df.columns]
        print(f"    celltypist_proba saved: {proba_df.shape}")

    adata_merged.uns["celltypist_direct"] = {
        "source_column":  src_key,
        "label_key":      CELLTYPIST_DIRECT_LABEL_KEY,
        "filtered_key":   CELLTYPIST_DIRECT_FILT_KEY,
        "conf_threshold": CELLTYPIST_CONF_THRESHOLD,
        "n_low_conf":     n_low,
    }


def run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION):
    print(f"\n[Leiden] Query-only clustering (resolution={resolution})...")
    qmask = adata_merged.obs["data_source"] == "query"
    qcells = adata_merged.obs_names[qmask]
    if len(qcells) < 10:
        adata_merged.obs["leiden_query"] = "N/A"; return
    X_q = adata_merged.obsm["X_scVI"][qmask.values]
    ad_ = sc.AnnData(X=X_q, obs=adata_merged.obs.loc[qcells].copy())
    ad_.obsm["X_scVI"] = X_q
    sc.pp.neighbors(ad_, use_rep="X_scVI", n_neighbors=30, random_state=RANDOM_SEED)
    sc.tl.leiden(ad_, resolution=resolution, random_state=RANDOM_SEED)
    lf = pd.Series("N/A", index=adata_merged.obs_names, dtype="object")
    lf.loc[qcells] = "qry_" + ad_.obs["leiden"].astype(str)
    adata_merged.obs["leiden_query"] = lf.values
    print(f"  -> {ad_.obs['leiden'].nunique()} query-only clusters")
    del ad_; gc.collect()


## Cell 3 — Initialization

In [4]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


Output directory: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline_v2


## Cell 4 — Step 1: Load

In [5]:
print("\n" + "=" * 80)
print("[Step 1] Loading...")
print("=" * 80)
t0 = time.time()
adata_ref = sc.read_h5ad(REFERENCE_H5AD); adata_ref.var_names_make_unique()
adata_qry = sc.read_h5ad(QUERY_H5AD);     adata_qry.var_names_make_unique()
print(f"[OK] Loaded in {time.time()-t0:.1f}s")
print(f"  Reference: {adata_ref.shape}")
print(f"  Query:     {adata_qry.shape}")



[Step 1] Loading...
[OK] Loaded in 303.6s
  Reference: (54743, 35112)
  Query:     (111272, 83690)


## Cell 5 — Step 2: Subset Reference (optional)

In [6]:
if INCLUDE_COARSE_TYPES is not None and "cell_type_L2" in adata_ref.obs.columns:
    mask = adata_ref.obs["cell_type_L2"].isin(INCLUDE_COARSE_TYPES)
    adata_ref = adata_ref[mask].copy()
    print(f"  Reference after subset: {adata_ref.shape}")
else:
    print("[Step 2] No subsetting.")


[Step 2] No subsetting.


## Cell 6 — Step 3: Preserve Reference Labels (QC only)

In [7]:
print("\n" + "=" * 80)
print("[Step 3] Preserving reference labels as QC columns (not used for scANVI training)...")
print("=" * 80)
# These columns are kept for comparison/QC only.
# scANVI will train on CellTypist labels for ALL cells.
if REF_LABEL_FINE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_fine_ref"] = adata_ref.obs[REF_LABEL_FINE].astype(str)
    print(f"  -> cell_type_fine_ref from: {REF_LABEL_FINE}")
    print(adata_ref.obs["cell_type_fine_ref"].value_counts())
else:
    adata_ref.obs["cell_type_fine_ref"] = "unknown"
    print(f"  [WARN] {REF_LABEL_FINE} not found in reference")

# Query has no prior labels — placeholder only
adata_qry.obs["cell_type_fine_ref"] = "N/A"



[Step 3] Preserving reference labels as QC columns (not used for scANVI training)...
  -> cell_type_fine_ref from: cell_type_L3
cell_type_fine_ref
0    22724
1    17585
2    10651
3     3763
4       20
Name: count, dtype: int64


## Cell 7 — Step 4: Common Genes

In [8]:
print("\n" + "=" * 80)
print("[Step 4] Finding common genes...")
print("=" * 80)
qry_set      = set(adata_qry.var_names)
common_genes = [g for g in adata_ref.var_names if g in qry_set]
print(f"  Ref: {adata_ref.n_vars:,}  Query: {adata_qry.n_vars:,}  Common: {len(common_genes):,}")
if len(common_genes) < 1000:
    raise ValueError(f"Too few common genes ({len(common_genes)})")
adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()



[Step 4] Finding common genes...
  Ref: 35,112  Query: 83,690  Common: 33,559


## Cell 8 — Step 5: Validate Counts

In [9]:
print("\n" + "=" * 80)
print("[Step 5] Validating counts...")
print("=" * 80)
ensure_counts_layer(adata_ref, "counts")
ensure_counts_layer(adata_qry, "counts")



[Step 5] Validating counts...
  [WARN] layers['counts'] not found, checking .X...
  -> Auto-copying .X to layers['counts']


'counts'

## Cell 9 — Step 6: Concatenate

In [10]:
print("\n" + "=" * 80)
print("[Step 6] Concatenating (ref + query symmetrically)...")
print("=" * 80)

# Prefix BATCH_KEY to prevent sample-name collisions across datasets.
# Use astype("string").fillna() so NA values never become literal "nan".
for ad, prefix in [(adata_ref, "ref_"), (adata_qry, "qry_")]:
    if BATCH_KEY not in ad.obs.columns:
        print(f"  [WARN] {BATCH_KEY} missing, creating placeholder")
        ad.obs[BATCH_KEY] = "unknown_batch"
    ad.obs[BATCH_KEY] = prefix + ad.obs[BATCH_KEY].astype("string").fillna("unknown_batch")

adata_ref.obs_names = pd.Index([f"ref_{x}" for x in adata_ref.obs_names])
adata_qry.obs_names = pd.Index([f"qry_{x}" for x in adata_qry.obs_names])

adata_merged = sc.concat(
    {"reference": adata_ref, "query": adata_qry},
    axis=0, join="inner", merge="unique", label="data_source"
)
print(f"  [INFO] Merged: {adata_merged.shape}")
print(f"  Reference: {(adata_merged.obs['data_source']=='reference').sum():,}")
print(f"  Query:     {(adata_merged.obs['data_source']=='query').sum():,}")

del adata_ref, adata_qry
gc.collect()



[Step 6] Concatenating (ref + query symmetrically)...
  [INFO] Merged: (166015, 33559)
  Reference: 54,743
  Query:     111,272


67457

## Cell 10 — Step 7: Covariates & symbol_base

In [11]:
print("\n" + "=" * 80)
print("[Step 7] Preparing covariates...")
print("=" * 80)
t0 = time.time()
prepare_covariates(adata_merged)
print(f"[OK] Covariates done in {time.time()-t0:.1f}s")

print("  -> Adding symbol_base column...")
adata_merged.var["symbol_base"] = adata_merged.var_names.str.replace(r"-\d+$", "", regex=True)



[Step 7] Preparing covariates...
  -> Validating counts...
  -> Batch/Tissue...
  -> Signature scores...
  -> Computing monocyte subtype scores...
    Classical: 14053
    NonClassical: 50629
    Intermediate: 22786
    DoublePositive: 2696
    Negative: 75851
[OK] Covariates done in 18.7s
  -> Adding symbol_base column...


## Cell 11 — Step 8: CellTypist on ALL Cells
> CellTypist annotates ref AND query cells with equal weight.
> These labels will be used as scANVI training labels for everyone.

In [12]:
print("\n" + "=" * 80)
print("[Step 8] Running CellTypist on all merged cells...")
print("=" * 80)
import time as _time
_t8 = _time.time()

celltypist_predictions = run_celltypist_on_full_genes(adata_merged)
export_celltypist_direct_results(adata_merged, celltypist_predictions)

print(f"[OK] CellTypist done in {_time.time()-_t8:.1f}s")
print()

# ── Label distribution (all cells) ───────────────────────────────────────────
_ct_counts = adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].value_counts()
n_unknown   = (adata_merged.obs[CELLTYPIST_DIRECT_FILT_KEY] == UNLABELED_CATEGORY).sum()

print(f"[INFO] Total cells: {adata_merged.n_obs:,}")
print(f"[INFO] Unique labels (direct):    {_ct_counts.shape[0]}")
print(f"[INFO] Unknown after conf filter: {n_unknown:,} ({n_unknown/adata_merged.n_obs:.1%})")
print()
print("[INFO] Label distribution (all cells):")
for _lbl, _n in _ct_counts.items():
    print(f"  {_lbl:<44} {_n:>7,}  ({_n/adata_merged.n_obs*100:.1f}%)")

# ── Lineage summary (myeloid-specific) ───────────────────────────────────────
_MYELOID_LINEAGE = {
    "Classical monocytes":        "Monocyte",
    "Non-classical monocytes":    "Monocyte",
    "Intermediate macrophages":   "Monocyte",
    "Monocytes":                  "Monocyte",
    "Macrophages":                "Macrophage",
    "Alveolar macrophages":       "Macrophage",
    "Intestinal macrophages":     "Macrophage",
    "DC1":                        "DC",
    "DC2":                        "DC",
    "pDC":                        "DC",
    "Migratory DCs":              "DC",
    "Mast cells":                 "Mast",
    "Regulatory T cells":         "contaminant_T",
    "Tcm/Naive helper T cells":   "contaminant_T",
    "Tem/Effector helper T cells":"contaminant_T",
    "Plasma cells":               "contaminant_B",
    "Epithelial cells":           "contaminant_epi",
}
_lineage_counts = {}
for _lbl, _n in _ct_counts.items():
    _lin = _MYELOID_LINEAGE.get(_lbl, "other")
    _lineage_counts[_lin] = _lineage_counts.get(_lin, 0) + _n

print()
print("[INFO] Lineage summary:")
for _lin, _n in sorted(_lineage_counts.items(), key=lambda x: -x[1]):
    _flag = "  [WARN] CONTAMINANT" if "contaminant" in _lin else ""
    print(f"  {_lin:<22} {_n:>7,}  ({_n/adata_merged.n_obs*100:.1f}%){_flag}")

_contam_total = sum(v for k, v in _lineage_counts.items() if "contaminant" in k)
if _contam_total > 0:
    print()
    print(f"  [WARN] Total putative contaminants: {_contam_total:,} "
          f"({_contam_total/adata_merged.n_obs:.1%}) — consider filtering before scANVI")

print()
print("[INFO] Label distribution by data_source:")
for _src in adata_merged.obs["data_source"].unique():
    _mask = adata_merged.obs["data_source"] == _src
    _sub  = adata_merged.obs.loc[_mask, CELLTYPIST_DIRECT_LABEL_KEY].value_counts()
    print(f"  [{_src}]  n={_mask.sum():,}")
    for _lbl, _n in _sub.items():
        print(f"    {_lbl:<44} {_n:>7,}  ({_n/_mask.sum()*100:.1f}%)")
    print()



[Step 8] Running CellTypist on all merged cells...

[CellTypist] Annotating ALL cells (ref + query)...
  -> Loaded from: /home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl
  -> 6064/6639 genes matched via symbol_base


🔬 Input data has 166015 cells and 6064 genes
🔗 Matching reference genes in the model
🧬 6064 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


  -> CellTypist complete:
Mast cells                  42817
Classical monocytes         38679
Alveolar macrophages        23612
Macrophages                 16792
Monocytes                    5814
Regulatory T cells           5701
DC2                          5513
Intermediate macrophages     4328
pDC                          3837
Non-classical monocytes      3118
Name: count, dtype: int64
  -> Exporting CellTypist direct branch...
    celltypist_label_direct: 17 types
    celltypist_label_direct_filt:  35460 low-conf cells -> 'Unknown'
    celltypist_proba saved: (166015, 98)
[OK] CellTypist done in 515.0s

[INFO] Total cells: 166,015
[INFO] Unique labels (direct):    17
[INFO] Unknown after conf filter: 35,460 (21.4%)

[INFO] Label distribution (all cells):
  Classical monocytes                           54,173  (32.6%)
  Mast cells                                    43,416  (26.2%)
  Alveolar macrophages                          25,727  (15.5%)
  Macrophages                          

## Cell 12 — Step 9: HVG Selection

In [13]:
print("\n" + "=" * 80)
print("[Step 9] Selecting HVGs...")
print("=" * 80)
hvg_method = "unknown"
try:
    sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                batch_key=BATCH_KEY, flavor="seurat_v3", subset=False)
    hvg_method = "batch_seurat_v3"
except Exception as e1:
    print(f"  -> batch-aware failed ({str(e1)[:50]}), trying standard...")
    try:
        sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                    flavor="seurat_v3", subset=False)
        hvg_method = "standard_seurat_v3"
    except Exception as e2:
        print(f"  -> seurat_v3 failed ({str(e2)[:50]}), fallback to cell_ranger")
        sc.pp.highly_variable_genes(adata_merged, layer="counts", n_top_genes=N_HVG,
                                    flavor="cell_ranger", subset=False)
        hvg_method = "cell_ranger"
print(f"  -> Method: {hvg_method}")

if FORCE_MARKERS_IN_HVG:
    n_added = 0
    mset    = set(FORCED_MARKERS)
    for idx, sb in enumerate(adata_merged.var["symbol_base"]):
        if sb in mset:
            rn = adata_merged.var_names[idx]
            if not adata_merged.var.loc[rn, "highly_variable"]:
                adata_merged.var.loc[rn, "highly_variable"] = True
                n_added += 1
    print(f"  -> Forced {n_added} markers into HVG")

n_hvg_final = int(adata_merged.var["highly_variable"].sum())
print(f"  -> Final HVG count: {n_hvg_final}")

hvg_genes = adata_merged.var_names[adata_merged.var["highly_variable"]].tolist()
(output_dir / f"{OUTPUT_PREFIX}_hvg_genes.txt").write_text("\n".join(hvg_genes))



[Step 9] Selecting HVGs...
  -> batch-aware failed (b'There are other near singularities as well. 0.09), trying standard...
  -> Method: standard_seurat_v3
  -> Forced 15 markers into HVG
  -> Final HVG count: 4015


30926

## Cell 13 — Step 10: Build Full Matrix for .raw

In [14]:
print("\n" + "=" * 80)
print("[Step 10] Building full matrix for .raw...")
print("=" * 80)
full_counts = adata_merged.layers["counts"]
if issparse(full_counts) and not isinstance(full_counts, csr_matrix):
    full_counts = csr_matrix(full_counts)
raw_var = adata_merged.var.copy()
print(f"  Full matrix: {full_counts.shape}")

# Checkpoint
adata_merged.write_h5ad(
    str(output_dir / f"{OUTPUT_PREFIX}_adata_preprocessed.h5ad"),
    compression="gzip"
)
print(f"[OK] Checkpoint saved: {OUTPUT_PREFIX}_adata_preprocessed.h5ad")



[Step 10] Building full matrix for .raw...
  Full matrix: (166015, 33559)


... storing 'cell_type_fine_ref' as categorical
... storing 'mono_subtype_by_score' as categorical
... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'symbol_base' as categorical


[OK] Checkpoint saved: myeloid_merged_v2_adata_preprocessed.h5ad


## Cell 14 — Step 11: Build Training Subset with CellTypist Labels for ALL Cells
> **Key change from v1.x:** `scanvi_labels` comes from CellTypist for BOTH ref and query.
> Query cells are no longer forced to `Unknown`.
> Only genuinely low-confidence cells (< threshold) become `Unknown`.

In [15]:
print("\n" + "=" * 80)
print("[Step 11] Building training subset (HVG only)...")
print("=" * 80)

hvg_mask = adata_merged.var["highly_variable"].values
X_hvg    = adata_merged.layers["counts"][:, hvg_mask]
if issparse(X_hvg) and not isinstance(X_hvg, csr_matrix):
    X_hvg = csr_matrix(X_hvg)

adata_train = sc.AnnData(
    X=X_hvg.copy(),
    obs=adata_merged.obs.copy(),
    var=adata_merged.var.iloc[hvg_mask].copy()
)
adata_train.var_names        = adata_merged.var_names[hvg_mask]
adata_train.layers["counts"] = adata_train.X
print(f"  [INFO] Training data: {adata_train.shape}")

# =============================================================================
# ARCHITECTURE v2.0: scanvi_labels from CellTypist for ALL cells equally.
# SCANVI_LABEL_SOURCE = "filtered" -> use celltypist_label_direct_filt
#                                     (low-conf cells already set to Unknown)
# SCANVI_LABEL_SOURCE = "direct"   -> use celltypist_label_direct (unfiltered)
# =============================================================================
label_source_col = (
    CELLTYPIST_DIRECT_FILT_KEY if SCANVI_LABEL_SOURCE == "filtered"
    else CELLTYPIST_DIRECT_LABEL_KEY
)
print(f"  -> scANVI label source: {label_source_col} (SCANVI_LABEL_SOURCE='{SCANVI_LABEL_SOURCE}')")
print(f"     All cells use CellTypist labels — ref and query treated equally")

adata_train.obs["scanvi_labels"] = adata_train.obs[label_source_col].astype(str)
adata_train.obs["scanvi_labels"] = adata_train.obs["scanvi_labels"].astype("category")
if UNLABELED_CATEGORY not in adata_train.obs["scanvi_labels"].cat.categories:
    adata_train.obs["scanvi_labels"] = adata_train.obs["scanvi_labels"].cat.add_categories([UNLABELED_CATEGORY])

print("\n  -> scANVI training label distribution:")
print(adata_train.obs["scanvi_labels"].value_counts())
n_unknown = (adata_train.obs["scanvi_labels"] == UNLABELED_CATEGORY).sum()
print(f"\n  -> Unknown (low-conf) cells: {n_unknown:,} / {adata_train.n_obs:,} "
      f"({100*n_unknown/adata_train.n_obs:.1f}%)")

gc.collect()



[Step 11] Building training subset (HVG only)...
  [INFO] Training data: (166015, 4015)
  -> scANVI label source: celltypist_label_direct_filt (SCANVI_LABEL_SOURCE='filtered')
     All cells use CellTypist labels — ref and query treated equally

  -> scANVI training label distribution:
scanvi_labels
Mast cells                     41874
Classical monocytes            36719
Unknown                        35460
Alveolar macrophages           23053
Macrophages                    13062
DC2                             4650
pDC                             3625
Non-classical monocytes         2258
Intermediate macrophages        1874
Intestinal macrophages           927
Monocytes                        793
Regulatory T cells               543
DC1                              507
Epithelial cells                 384
Migratory DCs                    127
Tcm/Naive helper T cells         111
Tem/Effector helper T cells       36
Plasma cells                      12
Name: count, dtype: int64

  -> 

20

## Cell 15 — Step 12: scVI Training

In [16]:
print("\n" + "=" * 80)
print("[Step 12] Training scVI...")
print("=" * 80)

scvi.model.SCVI.setup_anndata(
    adata_train,
    layer="counts",
    batch_key=BATCH_KEY,
    continuous_covariate_keys=["pct_counts_mt","stress_score","S_score","G2M_score"],
    categorical_covariate_keys=[TISSUE_KEY]
)

scvi_model = scvi.model.SCVI(
    adata_train,
    n_latent=SCVI_N_LATENT, n_layers=SCVI_N_LAYERS,
    n_hidden=SCVI_N_HIDDEN, dropout_rate=SCVI_DROPOUT
)

train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 30,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    train_kwargs["accelerator"] = "gpu"; train_kwargs["devices"] = 1

t0 = time.time()
scvi_model.train(**train_kwargs)
print(f"[OK] scVI training complete  ({time.time()-t0:.1f}s)")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



[Step 12] Training scVI...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 400/400: 100%|██████████| 400/400 [2:14:36<00:00, 19.76s/it, v_num=1, train_loss_step=669, train_loss_epoch=673]  

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [2:14:36<00:00, 20.19s/it, v_num=1, train_loss_step=669, train_loss_epoch=673]
[OK] scVI training complete  (8077.1s)


## Cell 16 — Step 13: scANVI Training

In [17]:
print("\n" + "=" * 80)
print("[Step 13] Training scANVI (CellTypist labels for all cells)...")
print("=" * 80)

scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model, adata=adata_train,
    labels_key="scanvi_labels",
    unlabeled_category=UNLABELED_CATEGORY
)

scanvi_train_kwargs = {
    "max_epochs": MAX_EPOCHS_SCANVI, "batch_size": BATCH_SIZE,
    "early_stopping": True, "early_stopping_patience": 20,
    "plan_kwargs": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    scanvi_train_kwargs["accelerator"] = "gpu"; scanvi_train_kwargs["devices"] = 1

t0 = time.time()
scanvi_model.train(**scanvi_train_kwargs)
print(f"[OK] scANVI training complete  ({time.time()-t0:.1f}s)")



[Step 13] Training scANVI (CellTypist labels for all cells)...
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 138/200:  69%|██████▉   | 138/200 [1:38:06<44:04, 42.66s/it, v_num=1, train_loss_step=682, train_loss_epoch=656]  
Monitored metric elbo_validation did not improve in the last 20 records. Best score: 689.529. Signaling Trainer to stop.
[OK] scANVI training complete  (5887.8s)


## Cell 17 — Step 14: Export Latent & Predictions

In [18]:
print("\n" + "=" * 80)
print("[Step 14] Exporting latent representations and predictions...")
print("=" * 80)

# scANVI latent (index-aligned via reindex)
lat     = scanvi_model.get_latent_representation(adata_train)
lat_df  = pd.DataFrame(lat, index=adata_train.obs_names,
                        columns=[f"scANVI_{i}" for i in range(lat.shape[1])])
lat_al  = lat_df.reindex(adata_merged.obs_names)
if lat_al.isna().any().any():
    raise ValueError("Missing scANVI latent after reindex!")
adata_merged.obsm["X_scANVI"] = lat_al.values

# Hard labels
pred_al = pd.Series(scanvi_model.predict(adata_train), index=adata_train.obs_names
                    ).reindex(adata_merged.obs_names)
adata_merged.obs["scanvi_pred"] = pred_al.values

# Soft probabilities — label_order from DataFrame columns (not scanvi_model.labels_)
proba_raw = scanvi_model.predict(adata_train, soft=True)
if isinstance(proba_raw, pd.DataFrame):
    label_order = list(proba_raw.columns)
    proba       = proba_raw.values.astype(np.float32)
else:
    proba = np.asarray(proba_raw, dtype=np.float32)
    try:
        label_order = list(scanvi_model.adata_manager.get_state_registry("labels").categorical_mapping)
    except Exception:
        label_order = [f"label_{i}" for i in range(proba.shape[1])]

proba_df = pd.DataFrame(proba, index=adata_train.obs_names, columns=label_order)
proba_al = proba_df.reindex(adata_merged.obs_names)
adata_merged.obsm["scanvi_proba"]      = proba_al.values
adata_merged.obs["scanvi_confidence"]  = proba_al.values.max(axis=1)
adata_merged.uns["scanvi_label_order"] = list(label_order)

print("  -> scANVI predictions (top 15):")
print(adata_merged.obs["scanvi_pred"].value_counts().head(15))

# Novelty score (entropy of scANVI probability distribution)
if not np.isfinite(proba_al.values).all():
    raise ValueError("Non-finite values in scANVI probability matrix!")
ents = entropy(proba_al.values + 1e-10, axis=1)
adata_merged.obs["scanvi_entropy"] = ents
emin, emax = ents.min(), ents.max()
adata_merged.obs["novelty_score"] = (ents - emin)/(emax - emin) if emax > emin else 0.0
qmask = adata_merged.obs["data_source"] == "query"
adata_merged.obs["is_potentially_novel"] = (adata_merged.obs["novelty_score"] > 0.7) & qmask
print(f"  -> High novelty query cells: {adata_merged.obs['is_potentially_novel'].sum()}")



[Step 14] Exporting latent representations and predictions...
  -> scANVI predictions (top 15):
scanvi_pred
Classical monocytes         68074
Mast cells                  56500
Macrophages                 22624
Alveolar macrophages         8747
DC2                          4728
pDC                          4035
Non-classical monocytes      1074
Intermediate macrophages      172
DC1                            22
Epithelial cells               16
Migratory DCs                  14
Intestinal macrophages          9
Name: count, dtype: int64
  -> High novelty query cells: 1203


## Cell 18 — Step 15: Attach .raw

In [19]:
print("\n" + "=" * 80)
print("[Step 15] Attaching .raw...")
print("=" * 80)
from anndata import AnnData
adata_merged.raw = AnnData(X=full_counts, obs=adata_merged.obs.copy(), var=raw_var)
print(f"  [OK] .raw: {adata_merged.raw.n_vars} genes")



[Step 15] Attaching .raw...
  [OK] .raw: 33559 genes


## Cell 19 — Step 16: Multiple UMAPs

In [20]:
print("\n" + "=" * 80)
print("[Step 16] Computing UMAPs...")
print("=" * 80)

# scVI latent
lat_s   = scvi_model.get_latent_representation(adata_train)
lat_s_df = pd.DataFrame(lat_s, index=adata_train.obs_names,
                         columns=[f"scVI_{i}" for i in range(lat_s.shape[1])])
lat_s_al = lat_s_df.reindex(adata_merged.obs_names)
if lat_s_al.isna().any().any():
    raise ValueError("Missing scVI latent after reindex!")
adata_merged.obsm["X_scVI"] = lat_s_al.values

run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION)

# scVI UMAP
sc.pp.neighbors(adata_merged, use_rep="X_scVI",   n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scVI")
adata_merged.obsm["X_umap_scVI"] = adata_merged.obsm["X_umap"].copy()
print("  -> X_umap_scVI saved")

# scANVI UMAP (DEFAULT)
sc.pp.neighbors(adata_merged, use_rep="X_scANVI", n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scANVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scANVI")
adata_merged.obsm["X_umap_scANVI"] = adata_merged.obsm["X_umap"].copy()
adata_merged.obsm["X_umap"]        = adata_merged.obsm["X_umap_scANVI"].copy()
print("  -> X_umap_scANVI saved (default X_umap)")

umap_op = UMAP(n_neighbors=30, n_components=2, min_dist=0.5, metric="euclidean",
               random_state=RANDOM_SEED)
umap_op.fit(adata_merged.obsm["X_scANVI"])
joblib.dump(umap_op, output_dir / f"{OUTPUT_PREFIX}_umap_scanvi_operator.joblib")
print("  -> UMAP operator saved")



[Step 16] Computing UMAPs...

[Leiden] Query-only clustering (resolution=1.0)...
  -> 18 query-only clusters
  -> X_umap_scVI saved
  -> X_umap_scANVI saved (default X_umap)
  -> UMAP operator saved


## Cell 20 — Step 16.5: Run Log

In [21]:
print("\n" + "=" * 80)
print("[Step 16.5] Writing run log...")
print("=" * 80)
from scipy.sparse import issparse as _issparse

log_ts    = datetime.now().isoformat()
out_h5ad  = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"

def _ms(mapping):
    return {str(k): {"shape": list(getattr(v,"shape",[])),"dtype": str(getattr(v,"dtype",""))}
            for k in sorted(mapping.keys()) for v in [mapping[k]]}

pipeline_log = {
    "version": "2.0", "timestamp": log_ts,
    "reference_h5ad": REFERENCE_H5AD, "query_h5ad": QUERY_H5AD,
    "output_dir": str(output_dir), "output_prefix": OUTPUT_PREFIX,
    "planned_output_h5ad": str(out_h5ad),
    "elapsed_min": round((time.time() - PIPELINE_START) / 60, 2),
    "n_obs": int(adata_merged.n_obs), "n_vars": int(adata_merged.n_vars),
    "n_hvg": n_hvg_final, "gpu_available": bool(gpu_available),
    "architecture": "CellTypist-equal scANVI (no ref/query asymmetry in labels)",
    "scanvi_label_source": label_source_col,
}
anndata_structure = {
    "timestamp": log_ts, "shape": [int(adata_merged.n_obs), int(adata_merged.n_vars)],
    "raw": {"present": adata_merged.raw is not None,
            "shape": list(adata_merged.raw.shape) if adata_merged.raw else None},
    "obs_columns": list(adata_merged.obs.columns),
    "obsm_keys":   sorted(adata_merged.obsm.keys()),
    "uns_keys":    sorted(str(k) for k in adata_merged.uns.keys()),
}
adata_merged.uns["pipeline_log"]      = pipeline_log
adata_merged.uns["anndata_structure"] = anndata_structure

log_lines = [f"{k}: {v}" for k,v in pipeline_log.items()]
(output_dir / f"{OUTPUT_PREFIX}_run_log.txt").write_text("\n".join(log_lines), encoding="utf-8")
with open(output_dir / f"{OUTPUT_PREFIX}_anndata_structure.json", "w") as f:
    json.dump(anndata_structure, f, indent=2)
print(f"  -> Log written")



[Step 16.5] Writing run log...
  -> Log written


## Cell 21 — Step 17: Save

In [22]:
print("\n" + "=" * 80)
print("[Step 17] Saving...")
print("=" * 80)

# Convert label columns to category before write_h5ad
_cat_cols = [
    "data_source", "cell_type_fine_ref",
    "scanvi_labels", "scanvi_pred",
    "mono_subtype_by_score", "leiden_query",
    CELLTYPIST_DIRECT_LABEL_KEY, CELLTYPIST_DIRECT_FILT_KEY,
]
for col in _cat_cols:
    for ad in [adata_merged, adata_train]:
        if col in ad.obs.columns:
            ad.obs[col] = ad.obs[col].astype("category")

scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model", overwrite=True)
scvi_model.save(output_dir  / f"{OUTPUT_PREFIX}_scvi_model",   overwrite=True)

config = {
    "version":    "2.0",
    "timestamp":  datetime.now().isoformat(),
    "input":      {"reference": REFERENCE_H5AD, "query": QUERY_H5AD},
    "n_hvg":      n_hvg_final,
    "architecture": {
        "description":    "CellTypist-equal scANVI — all cells labeled by CellTypist",
        "scanvi_labels":  label_source_col,
        "ref_labels_col": "cell_type_fine_ref (QC only, not used for training)",
    },
    "annotation_layers": {
        "celltypist_direct": {
            "label_key":      CELLTYPIST_DIRECT_LABEL_KEY,
            "filtered_key":   CELLTYPIST_DIRECT_FILT_KEY,
            "proba_key":      "celltypist_proba",
            "conf_threshold": CELLTYPIST_CONF_THRESHOLD,
        },
        "scanvi": {
            "pred_key":  "scanvi_pred",
            "conf_key":  "scanvi_confidence",
            "latent_key":"X_scANVI",
            "proba_key": "scanvi_proba",
        },
    },
    "umap_spaces": {
        "X_umap":        "DEFAULT (scANVI-based)",
        "X_umap_scVI":   "scVI latent UMAP",
        "X_umap_scANVI": "scANVI latent UMAP",
    },
    "scanvi_labels": list(label_order),
}

with open(output_dir / f"{OUTPUT_PREFIX}_config.json", "w") as f:
    json.dump(config, f, indent=2)


# Annotation statistics CSV (mirrors epithelial pipeline output format)
_stats = (
    adata_merged.obs["scanvi_pred"].value_counts()
    .rename_axis("Cell_Type").reset_index(name="Count")
)
_stats["Percentage"] = 100 * _stats["Count"] / _stats["Count"].sum()
_conf = adata_merged.obs.groupby("scanvi_pred")["scanvi_confidence"].agg(["mean", "std"])
_stats = _stats.merge(_conf, left_on="Cell_Type", right_index=True, how="left")
_stats.to_csv(output_dir / f"{OUTPUT_PREFIX}_scanvi_statistics.csv", index=False)

# Per data_source breakdown
adata_merged.obs[["data_source", "scanvi_pred",
                   CELLTYPIST_DIRECT_LABEL_KEY, CELLTYPIST_DIRECT_FILT_KEY,
                   "scanvi_confidence", "novelty_score"]].to_csv(
    output_dir / f"{OUTPUT_PREFIX}_annotations.csv"
)
print(f"[OK] Annotation CSVs saved")
adata_merged.write_h5ad(out_h5ad, compression="gzip")
print(f"  -> {out_h5ad}")
adata_train.write_h5ad(output_dir / f"{OUTPUT_PREFIX}_train_HVG.h5ad", compression="gzip")
print(f"  -> train HVG saved")



[Step 17] Saving...
[OK] Annotation CSVs saved


... storing 'symbol_base' as categorical


  -> /home/h2048/data/py/20260308/myeloid_only_merged_pipeline_v2/myeloid_merged_v2_results.h5ad
  -> train HVG saved


## Cell 22 — Step 18: Visualization

In [25]:
print("\n" + "=" * 80)
print("[Step 18] Visualization...")
elapsed = (time.time() - PIPELINE_START) / 60
print("=" * 80)
print(f"PIPELINE COMPLETE  |  Elapsed: {elapsed:.1f} min")

qmask = adata_merged.obs["data_source"] == "query"

# ── Panel 1: Overview 4x4 ─────────────────────────────────────────────────
fig = plt.figure(figsize=(24, 20))
gs  = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

panels_row0 = [
    ("data_source",              "Data Source",         None,     None),
    ("cell_type_fine_ref",       "Ref Original Labels", None,     None),
    (CELLTYPIST_DIRECT_LABEL_KEY,"CellTypist Direct",   None,     None),
    (CELLTYPIST_DIRECT_FILT_KEY, f"CellTypist Filtered (conf>={CELLTYPIST_CONF_THRESHOLD})", None, None),
]
panels_row1 = [
    ("scanvi_pred",       "scANVI Predictions",  None,      None),
    ("scanvi_confidence", "scANVI Confidence",   "viridis", (0,1)),
    ("novelty_score",     "Novelty Score",       "hot",     (0,1)),
    ("mono_subtype_by_score","Mono Subtype",     None,      None),
]

for col, (key, title, cmap, vlim) in enumerate(panels_row0):
    ax = fig.add_subplot(gs[0, col])
    kw = dict(ax=ax, show=False, title=title, s=15, legend_loc="on data")
    if cmap:
        kw["cmap"] = cmap
    if vlim:
        kw["vmin"], kw["vmax"] = vlim
    if key in adata_merged.obs.columns:
        sc.pl.umap(adata_merged, color=key, **kw)

for col, (key, title, cmap, vlim) in enumerate(panels_row1):
    ax = fig.add_subplot(gs[1, col])
    kw = dict(ax=ax, show=False, title=title, s=15)
    if cmap:
        kw["cmap"] = cmap
    if vlim:
        kw["vmin"], kw["vmax"] = vlim
    if key not in adata_merged.obs.columns:
        continue
    if cmap:
        sc.pl.umap(adata_merged, color=key, **kw)
    else:
        sc.pl.umap(adata_merged, color=key, legend_loc="on data", **kw)

# Marker gene panels
marker_panels = [
    ("LYZ","LYZ (pan-Myeloid)"),("CD14","CD14 (Classical Mono)"),
    ("FCGR3A","FCGR3A (NonClassical)"),("CD68","CD68 (Macro)"),
    ("CD1C","CD1C (cDC2)"),("CLEC9A","CLEC9A (cDC1)"),
    ("S100A8","S100A8 (Neutrophil)"),("MKI67","MKI67 (Prolif)"),
]
for i, (gene, title) in enumerate(marker_panels):
    row, col_i = 2 + i//4, i%4
    ax = fig.add_subplot(gs[row, col_i])
    if adata_merged.raw is not None and gene in adata_merged.raw.var_names:
        sc.pl.umap(
            adata_merged,
            color=gene,
            ax=ax,
            show=False,
            title=title,
            cmap="Reds",
            s=15,
            use_raw=True,
        )
    else:
        ax.set_title(f"{title} (not found)")

plt.savefig(
    output_dir / f"{OUTPUT_PREFIX}_overview.pdf",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
    format=FIGURE_FORMAT,
    )
plt.close()
print(f"  -> {OUTPUT_PREFIX}_overview.pdf")

# ── Panel 2: CellTypist vs scANVI comparison ──────────────────────────────
compare_keys = [
    (CELLTYPIST_DIRECT_LABEL_KEY, "CellTypist Direct"),
    (CELLTYPIST_DIRECT_FILT_KEY,  f"CellTypist Filtered"),
    ("scanvi_pred",               "scANVI (trained on CellTypist labels)"),
]
fig2, axes2 = plt.subplots(1, 3, figsize=(21, 7))
for ax, (key, title) in zip(axes2, compare_keys):
    if key in adata_merged.obs.columns:
        sc.pl.umap(
            adata_merged,
            color=key,
            ax=ax,
            show=False,
            title=title,
            legend_loc="on data",
            s=10,
        )
plt.tight_layout()
plt.savefig(
    output_dir / f"{OUTPUT_PREFIX}_celltypist_vs_scanvi.pdf",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
    format=FIGURE_FORMAT,
    )
plt.close()
print(f"  -> {OUTPUT_PREFIX}_celltypist_vs_scanvi.pdf")

# ── Panel 3: scVI vs scANVI UMAP ──────────────────────────────────────────
fig3, axes3 = plt.subplots(2, 3, figsize=(18, 12))
for row, (basis, label) in enumerate([("X_umap_scVI","scVI"), ("X_umap_scANVI","scANVI")]):
    sc.pl.embedding(
        adata_merged,
        basis=basis,
        color="data_source",
        ax=axes3[row,0],
        show=False,
        title=f"Data Source ({label})",
        s=10,
    )
    sc.pl.embedding(
        adata_merged,
        basis=basis,
        color=CELLTYPIST_DIRECT_LABEL_KEY,
        ax=axes3[row,1],
        show=False,
        title=f"CellTypist Labels ({label})",
        legend_loc="on data",
        s=10,
    )
    sc.pl.embedding(
        adata_merged,
        basis=basis,
        color="scanvi_pred",
        ax=axes3[row,2],
        show=False,
        title=f"scANVI Pred ({label})",
        legend_loc="on data",
        s=10,
    )
plt.tight_layout()
plt.savefig(
    output_dir / f"{OUTPUT_PREFIX}_umap_comparison.pdf",
    dpi=FIGURE_DPI,
    bbox_inches="tight",
    format=FIGURE_FORMAT,
    )
plt.close()
print(f"  -> {OUTPUT_PREFIX}_umap_comparison.pdf")

# ── Summary ───────────────────────────────────────────────────────────────
summary_label_col = "scanvi_labels" if "scanvi_labels" in adata_merged.obs.columns else label_source_col
summary_unknown = 0
if summary_label_col in adata_merged.obs.columns:
    summary_unknown = int((adata_merged.obs[summary_label_col].astype(str) == UNLABELED_CATEGORY).sum())
elif "scanvi_labels" in adata_train.obs.columns:
    summary_unknown = int((adata_train.obs["scanvi_labels"].astype(str) == UNLABELED_CATEGORY).sum())

print("\n" + "="*80)
print("MYELOID PIPELINE v2.0 COMPLETE")
print("="*80)
print(f"  Total cells : {adata_merged.n_obs:,}  "
      f"(ref: {(adata_merged.obs['data_source']=='reference').sum():,}  "
      f"query: {(adata_merged.obs['data_source']=='query').sum():,})")
print(f"\nscANVI label source: {label_source_col}")
print(f"  Unknown (low-conf): {summary_unknown:,}")
print("\nTop scANVI predictions:")
print(adata_merged.obs["scanvi_pred"].value_counts().head(10))
print("\nTop CellTypist direct labels:")
print(adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].value_counts().head(10))
print("="*80)



[Step 18] Visualization...
PIPELINE COMPLETE  |  Elapsed: 467.6 min
  -> myeloid_merged_v2_overview.pdf
  -> myeloid_merged_v2_celltypist_vs_scanvi.pdf
  -> myeloid_merged_v2_umap_comparison.pdf

MYELOID PIPELINE v2.0 COMPLETE
  Total cells : 166,015  (ref: 54,743  query: 111,272)

scANVI label source: celltypist_label_direct_filt
  Unknown (low-conf): 35,460

Top scANVI predictions:
scanvi_pred
Classical monocytes         68074
Mast cells                  56500
Macrophages                 22624
Alveolar macrophages         8747
DC2                          4728
pDC                          4035
Non-classical monocytes      1074
Intermediate macrophages      172
DC1                            22
Epithelial cells               16
Name: count, dtype: int64

Top CellTypist direct labels:
celltypist_label_direct
Classical monocytes         54173
Mast cells                  43416
Alveolar macrophages        25727
Macrophages                 19543
DC2                          7168
pDC      